By the end, you’ll clearly understand:

JSON → Python → Structured Model
Validation (strict + coercion)
Nested objects (real enterprise payloads)
JSON output generation
Error handling (critical for your automation systems)

We will depend on Pydantic for auto type conversion instead of performing the conversion manually

Example 1: Using simple JSON data for input

In [1]:
# Defining a simple class
from pydantic import BaseModel

class ServerHealth(BaseModel):
    server_name: str
    cpu_usage: float
    memory_usage: float
    is_critical: bool

In [3]:
# Defining the Data Structure in JSON format
data = {
    "server_name": "app01",
    "cpu_usage": 95.0,
    "memory_usage": 92.5,
    "is_critical": True
}

In [4]:
# Passing the data to the ServerHealth Class
# here ** will unpack the dictionary data for the class parameters automatically
server = ServerHealth(**data)

In [5]:
# Getting the ouput from the Server
print(server)

server_name='app01' cpu_usage=95.0 memory_usage=92.5 is_critical=True


In [7]:
# Converting back the output into JSON format
print(server.model_dump_json(indent=2))

{
  "server_name": "app01",
  "cpu_usage": 95.0,
  "memory_usage": 92.5,
  "is_critical": true
}


Example 2: Working with Nested JSON Input

In this example we will define a class and will use the class inside another class (nested class)

Using Pydantic Field() and typing Annotated together

In [23]:
# Defining a class
from pydantic import BaseModel, Field, ValidationError, StringConstraints
from typing import Literal, Annotated


# Defining Server Class
class Server(BaseModel):
    name: str
    ip: str

# Defining the ticket priorities which are accepted
Ticket_Priorities = Literal["P1", "P2", "P3", "P4"]

# Defining reuseable type rules with Annotated
LowercasePrefix = Annotated[
    str,
    StringConstraints(to_lower=True, strip_whitespace=True)
]

# Defining Incident Class

class Incident(BaseModel):
    ticket_id: LowercasePrefix = Field(
        pattern = r'(?i)inc',
        description = "The field must start with 'INC' or 'Inc' or 'inc'. ",
        examples = ["inc890001"]
    )
    priority: Ticket_Priorities
    server: Server

In [24]:
# Defining the Data
data = {
    "ticket_id": "iNc8007890",
    "priority": "P2",
    "server": {
        "name": "webserver01",
        "ip": "10.10.1.10"
    }
}

In [25]:
# Create an Incident Object using the data

# Note: Incident Class will automatically call Server class constractor and will pass 
# 'name' and 'ip' to it to create the object

try:
    inc1 = Incident(**data)
    print("✅ Object created successfully:", inc1.ticket_id)
except ValidationError as ve:
    print("\n🔒 Invalid Data Type Supplied")
    print(ve.errors()[0]['msg'])

✅ Object created successfully: inc8007890


In [26]:
# Get the databack
print(inc1.model_dump_json(indent=2))

{
  "ticket_id": "inc8007890",
  "priority": "P2",
  "server": {
    "name": "webserver01",
    "ip": "10.10.1.10"
  }
}


Example 3: Creating an alert processor

1. Create JSON Alert
2. Parse using Pydantic
3. Validate rules
   CPU > 80 -> Critical
4. Output structured JSON


Note: Here we will use double nested class which will give you more deeper understanding on complex data handling

In [58]:
from pydantic import BaseModel, Field, ValidationError, StringConstraints, model_validator
from typing import Literal, Annotated
import ipaddress

Allowed_Incident_Types = Literal["P1", "P2", None]
LowercasePrefix = Annotated[
    str,
    StringConstraints(to_lower=True, strip_whitespace=True)
]

# ==========================================
# 1. Base Infrastructure Layer Models
# ==========================================
class Server(BaseModel):
    hostname: str
    ip: str

    @model_validator(mode='after')
    def validate_ip(self) -> 'Server':
        try:
            ipaddress.ip_address(self.ip.strip())
        except ValueError:
            raise ValueError(f"Invalid IP address format : {self.ip}")
        return self
    
# Class Alert
class CPU_Alert(BaseModel):
    server: Server # Of type Server class
    cpu_usage: float


# ==========================================
# 2. Incident Log Core Model
# ==========================================
class Incident(BaseModel):
    ticket_id: LowercasePrefix = Field(
        pattern = r'(?i)inc',
        description = "The field must start with 'INC' or 'Inc' or 'inc'. ",
        examples = ["inc890001"]
    )
    priority: Allowed_Incident_Types = None
    alert_type: Literal["CRITICAL", "HIGH", None] = None
    alert: CPU_Alert # of type CPU_Alert Class
    
    @model_validator(mode='after')
    def Set_Priority(self) -> 'Incident':
        usage = self.alert.cpu_usage

        if usage >= 92.0:
            self.alert_type = "CRITICAL"
            self.priority = "P1"
        elif 80.0 < usage < 92.0:
            self.alert_type = "CRITICAL"
            self.priority = "P1"
        else:
            self.priority = None
            self.alert_type = "NORMAL"
        
        return self
    
    def get_data(self) -> str:
        result = "\n".join([
            f"Ticket Number: {self.ticket_id}",
            f"Server Name : {self.alert.server.hostname}",
            f"CPU Usage: {self.alert.cpu_usage}%",
            f"Ticket Priority: {self.priority}",
            f"Alert Type: {self.alert_type}"
        ])

        return (result)
        



In [59]:
# ==========================================
# 3. Processing Orchestration Loop
# ==========================================
def process_incident(data):
    try:
        alert_obj = Incident(**data)
        print("✅ Incident Log Instantiated Successfully!")
    except ValidationError as ve:
        print("\n🔒 Invalid Data Type Supplied")
        print(ve.json(indent=2))
    
    return alert_obj

In [60]:
# JSON Data
data = {
   "ticket_id": "Inc7008976",
   "alert": {
        "server": {
            "hostname": "webserver1",
            "ip": "10.10.1.10"
        },
        "cpu_usage": 95.0
   }


}

In [61]:
# Creating the alert
print("Creating the Incident:")
alert_obj = process_incident(data)


Creating the Incident:
✅ Incident Log Instantiated Successfully!


In [62]:
# Getting the data
print("-" * 40)
print(alert_obj.get_data())

----------------------------------------
Ticket Number: inc7008976
Server Name : webserver1
CPU Usage: 95.0%
Ticket Priority: P1
Alert Type: CRITICAL
